# Warm-up 4B · Tết Demand Forecasting · Dự báo nhu cầu mua sắm mùa Tết (Time Series · Chuỗi thời gian)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | *Phố Xanh Mart*, a chain of 10 convenience stores in Hà Nội and TP.HCM, has to order its Tết stock **weeks in advance**. Using daily sales up to 31 Dec 2025, forecast the units sold (`sales`) of every item in every store, for every day from **1 Jan to 4 Mar 2026**. | Chuỗi cửa hàng tiện lợi *Phố Xanh Mart* (10 cửa hàng ở Hà Nội và TP.HCM) phải đặt hàng Tết **trước nhiều tuần**. Dựa vào doanh số hằng ngày đến hết 31/12/2025, hãy dự báo số lượng bán (`sales`) của từng mặt hàng ở từng cửa hàng, cho mỗi ngày từ **1/1 đến 4/3/2026**. |
| **Data** | `train/train.csv` 441,440 rows (store × item × day, 1 Nov 2022 – 31 Dec 2025) plus `items.csv`, `stores.csv` and `calendar.csv` (solar **and lunar** dates, public holidays, up to 4 Mar 2026). `public_test/` (14,110 rows) and `private_test/` (14,110 rows) both cover 1 Jan – 4 Mar 2026, each for 5 of the 10 stores. Prices and discounts of the test days are known in advance. | `train/train.csv` 441.440 dòng (cửa hàng × mặt hàng × ngày, 1/11/2022 – 31/12/2025) cùng `items.csv`, `stores.csv` và `calendar.csv` (ngày dương lịch **và âm lịch**, ngày nghỉ lễ, đến 4/3/2026). `public_test/` (14.110 dòng) và `private_test/` (14.110 dòng) cùng phủ giai đoạn 1/1 – 4/3/2026, mỗi tập gồm 5 trong 10 cửa hàng. Giá bán và khuyến mãi của các ngày test được biết trước. |
| **Metric** | Revenue-weighted WAPE (weight = the item's regular price, column `weight`), **lower is better**. Ranking uses the **private** score. | WAPE có trọng số doanh thu (trọng số = giá niêm yết của mặt hàng, cột `weight`), **càng thấp càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,sales` (non-negative numbers) | `public_submission.csv`, `private_submission.csv` gồm cột `id,sales` (số không âm) |
| **Rules** | PyTorch model, no external data. Calendar knowledge (public holidays, the lunar calendar) is fine, and it is already in `calendar.csv`. | Mô hình PyTorch, không dùng dữ liệu ngoài. Được dùng kiến thức về lịch (ngày lễ, âm lịch), và chúng đã có sẵn trong `calendar.csv`. |

$$\text{WAPE}=\frac{\sum_i w_i\,\lvert y_i-\hat y_i\rvert}{\sum_i w_i\,y_i}$$

🇬🇧 $y_i$ = units sold on row $i$, $w_i$ = its `weight` (regular price, thousand VND): the share of the revenue that the forecast gets wrong.
🇻🇳 $y_i$ = số lượng bán ở dòng $i$, $w_i$ = `weight` của dòng đó (giá niêm yết, nghìn đồng): tỷ lệ doanh thu bị dự báo sai.

**Columns · Các cột:**
- `train.csv`: `id, date, store_id, item_id, sales, sell_price, discount_pct` — `sell_price` = price paid that day (thousand VND) after the discount `discount_pct` · giá bán thực tế trong ngày (nghìn đồng) sau mức giảm giá `discount_pct`
- `public_test.csv`, `private_test.csv`: `id, date, store_id, item_id, sell_price, discount_pct, weight`
- `items.csv`: `item_id, item_name, category, regular_price` · `stores.csv`: `store_id, city, district, store_type, open_date`
- `calendar.csv`: `date`, `day_of_week` (0 = Monday · Thứ Hai), `lunar_day`, `lunar_month`, `lunar_leap_month`, `holiday`

🇬🇧 A row exists only when the item was on sale in that store: there are new items, a new store and Tết-only products. Zero sales can mean no demand **or an empty shelf**: in the past some items ran out of stock on some days. There were no stock-outs in the test period.
🇻🇳 Chỉ có dòng khi mặt hàng được bán ở cửa hàng đó: có mặt hàng mới, cửa hàng mới và hàng chỉ bán dịp Tết. Doanh số bằng 0 có thể do không ai mua **hoặc do kệ hết hàng**: trước đây có những ngày một số mặt hàng bị hết hàng. Giai đoạn test không có hết hàng.

Inspired by · Lấy cảm hứng từ: [Rohlik Sales Forecasting Challenge v2 (Kaggle, 2025)](https://www.kaggle.com/competitions/rohlik-sales-forecasting-challenge-v2) and [M5 Forecasting – Accuracy (Kaggle, 2020)](https://www.kaggle.com/competitions/m5-forecasting-accuracy).
The data here are synthetic · Dữ liệu ở đây là dữ liệu mô phỏng.

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "4B"
FOLDER  = "4B_tet_demand_forecast"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/warmup-4"

# Already have the folder? Point this at the parent of 4B_tet_demand_forecast/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 4B_tet_demand_forecast/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/4B_tet_demand_forecast/dataset"
TRAIN_CSV = f"{DATA_DIR}/train/train.csv"
ITEMS_CSV = f"{DATA_DIR}/train/items.csv"
STORES_CSV = f"{DATA_DIR}/train/stores.csv"
CALENDAR_CSV = f"{DATA_DIR}/train/calendar.csv"
PUBLIC_CSV = f"{DATA_DIR}/public_test/public_test.csv"
PRIVATE_CSV = f"{DATA_DIR}/private_test/private_test.csv"
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
TRAIN_FROM = "2023-01-01"  # rows used for training / dữ liệu dùng để huấn luyện
VAL_FROM = "2025-12-04"  # last 4 weeks = validation / 4 tuần cuối = validation
BATCH_SIZE = 2048
EPOCHS = 3
LR = 2e-3
HIDDEN = 64
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Features · Đọc dữ liệu và tạo đặc trưng

In [ ]:
train_df = pd.read_csv(TRAIN_CSV, parse_dates=["date"])
items = pd.read_csv(ITEMS_CSV)
stores = pd.read_csv(STORES_CSV)
calendar = pd.read_csv(CALENDAR_CSV, parse_dates=["date"])
public_df = pd.read_csv(PUBLIC_CSV, parse_dates=["date"])
private_df = pd.read_csv(PRIVATE_CSV, parse_dates=["date"])
print(f"Train: {len(train_df):,} rows, {train_df['date'].min().date()} .. {train_df['date'].max().date()}")
print(f"Public test: {len(public_df):,} rows | Private test: {len(private_df):,} rows")

TARGET = "sales"
PRICE = dict(zip(items["item_id"], items["regular_price"]))  # metric weight / trọng số trong độ đo
STORE_IDX = {s: i for i, s in enumerate(stores["store_id"])}
ITEM_IDX = {s: i for i, s in enumerate(items["item_id"])}


def make_features(df):
    """EN: ids (store, item, weekday, month, day of month) + the discount. Only the solar calendar is used.
    VI: mã (cửa hàng, mặt hàng, thứ, tháng, ngày trong tháng) + mức giảm giá. Chỉ dùng lịch dương."""
    cat = np.stack([df["store_id"].map(STORE_IDX), df["item_id"].map(ITEM_IDX), df["date"].dt.dayofweek,
                    df["date"].dt.month - 1, df["date"].dt.day - 1], axis=1)
    num = df[["discount_pct"]].to_numpy(dtype=np.float32)
    return torch.tensor(cat, dtype=torch.long), torch.tensor(num)


def make_loader(df, shuffle):
    """EN: target = log(1 + sales); test rows get a dummy 0. VI: nhãn = log(1 + doanh số); dòng test nhận 0."""
    cat, num = make_features(df)
    y = np.log1p(df[TARGET].to_numpy()) if TARGET in df else np.zeros(len(df))
    return DataLoader(TensorDataset(cat, num, torch.tensor(y, dtype=torch.float32)),
                      batch_size=BATCH_SIZE if shuffle else 8192, shuffle=shuffle)


# EN: time-based split: train on TRAIN_FROM .. VAL_FROM, validate on the last 4 weeks of 2025
# VI: tách theo thời gian: huấn luyện từ TRAIN_FROM đến VAL_FROM, kiểm tra trên 4 tuần cuối năm 2025
hist = train_df[train_df["date"] >= TRAIN_FROM]
tr_part, val_part = hist[hist["date"] < VAL_FROM], hist[hist["date"] >= VAL_FROM]
train_loader = make_loader(tr_part, shuffle=True)
val_loader = make_loader(val_part, shuffle=False)
public_loader = make_loader(public_df, shuffle=False)
private_loader = make_loader(private_df, shuffle=False)
print(f"Training rows: {len(tr_part):,} | Validation rows: {len(val_part):,} (from {VAL_FROM})")
train_df.head()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
class SalesMLP(nn.Module):
    """EN: embeddings of store, item, weekday, month and day of month + the discount -> MLP -> log(1 + sales).
    VI: embedding cửa hàng, mặt hàng, thứ, tháng, ngày trong tháng + mức giảm giá -> MLP -> log(1 + doanh số)."""

    def __init__(self, n_store, n_item, hidden=HIDDEN):
        super().__init__()
        self.store = nn.Embedding(n_store, 4)
        self.item = nn.Embedding(n_item, 8)
        self.dow = nn.Embedding(7, 3)
        self.month = nn.Embedding(12, 3)
        self.day = nn.Embedding(31, 3)
        self.mlp = nn.Sequential(nn.Linear(4 + 8 + 3 + 3 + 3 + 1, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, cat, num):
        x = torch.cat([self.store(cat[:, 0]), self.item(cat[:, 1]), self.dow(cat[:, 2]),
                       self.month(cat[:, 3]), self.day(cat[:, 4]), num], dim=1)
        return self.mlp(x).squeeze(-1)


model = SalesMLP(len(STORE_IDX), len(ITEM_IDX)).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
def wape(y_true, y_pred, weight):
    """EN: the competition metric (revenue-weighted WAPE). VI: độ đo của cuộc thi (WAPE theo doanh thu)."""
    return float(np.sum(weight * np.abs(y_true - y_pred)) / np.sum(weight * y_true))


@torch.no_grad()
def predict_sales(model, loader):
    """EN: forecasts in units (back from log(1 + sales)). VI: dự báo theo số lượng (đổi ngược từ log(1 + doanh số))."""
    model.eval()
    out = [model(cat.to(DEVICE), num.to(DEVICE)).cpu().numpy() for cat, num, _ in loader]
    return np.clip(np.expm1(np.concatenate(out)), 0, None)


criterion = nn.MSELoss()  # on log(1 + sales) / trên log(1 + doanh số)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
w_val = val_part["item_id"].map(PRICE).to_numpy()

best_wape = float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for cat, num, y in train_loader:
        cat, num, y = cat.to(DEVICE), num.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(cat, num), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)

    val_wape = wape(val_part[TARGET].to_numpy(), predict_sales(model, val_loader), w_val)
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(tr_part):.4f} | Val WAPE={val_wape:.4f}"
    if val_wape < best_wape:
        best_wape = val_wape
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation WAPE: {best_wape:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(ids, preds, csv_path):
    """Save <id, sales> CSV and zip it. / Lưu dự báo ra CSV và nén zip."""
    pd.DataFrame({"id": ids, TARGET: preds}).to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df["id"], predict_sales(model, public_loader), "public_submission.csv")
export(private_df["id"], predict_sales(model, private_loader), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", TARGET], f"{csv_path}: columns must be ['id', '{TARGET}']"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    assert sub[TARGET].notna().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"
    assert np.isfinite(sub[TARGET]).all(), f"{csv_path}: sales must be finite numbers / doanh số phải là số hữu hạn"
    assert (sub[TARGET] >= 0).all(), f"{csv_path}: sales must be >= 0 / doanh số phải >= 0"


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "4B"          # which problem this notebook is for
CONTEST = ""            # blank = the contest you are entered in
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")